# From Table to Network: the Python side

CIS 568 / DSC 530, Week 11. This notebook transforms the IMDB Top 1000 table into a movie similarity network and exports it as `output.json` for the D3 force layout in `index.html` / `network.js`.

Table → one-hot vectors → similarity matrix → network → JSON

In [ ]:
import json

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

datafile = "./imdb_top_1000.csv"

## Load the data

In [ ]:
df = pd.read_csv(datafile)
df.head()

In [ ]:
df.columns

## Step 1: Encode (one-hot)

Each distinct value in a column becomes its own 0/1 column. Every movie ends up with exactly six 1s, one per original column.

In [ ]:
# One 0/1 column per distinct value in each column
cols = ['Director', 'Genre', 'Star1', 'Star2', 'Star3', 'Star4']
df_hotbit = pd.get_dummies(df[cols], prefix='Category')

len(df_hotbit.columns)   # 4081

## Step 2: Compare (cosine similarity)

With six 1s per movie, cosine similarity is simply *shared attributes / 6*.

In [ ]:
similarity_matrix = cosine_similarity(df_hotbit)   # 1000 x 1000
similarity_matrix.shape

## Step 3: Filter into a network

A threshold of 0.3 keeps pairs of movies that share at least two attributes.

In [ ]:
threshold = 0.3
links = []
for i in range(len(similarity_matrix)):
    for j in range(i + 1, len(similarity_matrix)):   # upper triangle only
        if similarity_matrix[i][j] >= threshold:
            links.append({"source": i, "target": j,
                          "weight": round(similarity_matrix[i][j], 3)})

In [ ]:
# Keep only movies that appear in at least one link
nodes_set = {l["source"] for l in links} | {l["target"] for l in links}

nodes = [{"id": idx,
          "title":    df['Series_Title'].iloc[idx],
          "Director": df['Director'].iloc[idx],
          "Star1":    df['Star1'].iloc[idx],
          "Star2":    df['Star2'].iloc[idx],
          "Star3":    df['Star3'].iloc[idx],
          "Star4":    df['Star4'].iloc[idx],
          "Genre":    df['Genre'].iloc[idx],
          "poster":   df['Poster_Link'].iloc[idx]}
         for idx in sorted(nodes_set)]

len(nodes), len(links)   # (326, 323)

## Step 4: Export

`output.json` is the contract with the D3 side: two lists, `nodes` and `links`.

In [ ]:
with open('output.json', 'w') as f:
    json.dump({"links": links, "nodes": nodes}, f, indent=4)